In [ ]:
from visualization_utils import TruthData
import torch
from plotly.subplots import make_subplots
import configparser

model = 'llama-2-13b'
config = configparser.ConfigParser()
config.read('config.ini')
layer = eval(config[model]['probe_layer'])
noperiod = eval(config[model]['noperiod'])

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

In [ ]:
TruthData.from_datasets(
    ['human_farmed_neutral', 'human_farmed_harm'], # datasets to use
    model=model,
    layer=layer,
    center=True,
    noperiod=noperiod,
    device=device
).plot(
    dimensions=2, # 3 dimensions also supported
    dim_offset=0, # increase if you want to ignore the first few PCs
    color='label', # label 1 = first category in the dataset name (e.g. human)
    # if you don't want to plot and do PCA on all datasets:
    # plot_datasets = [ list of datasets to plot here ],
    # pca_datasets = [ list of datasets to use for PCA here]
)

In [ ]:
# visualization of all 12 animal/human datasets (label 1 = first category in the name)
datasets = [
    'human_companion_harm',
    'human_farmed_harm',
    'human_wild_harm',
    'companion_farmed_harm',
    'companion_wild_harm',
    'farmed_wild_harm',
    'human_companion_neutral',
    'human_farmed_neutral',
    'human_wild_neutral',
    'companion_farmed_neutral',
    'companion_wild_neutral',
    'farmed_wild_neutral',
]

td = TruthData.from_datasets(
    datasets,
    model=model,
    layer=layer,
    center=True,
    noperiod=noperiod,
    device=device)

fig = make_subplots(rows=3, cols=4, subplot_titles=datasets, horizontal_spacing=0.05, vertical_spacing=0.05)
for i, dataset in enumerate(datasets):
    for data in td.plot(
        dimensions=2,
        color='label',
        plot_datasets = [dataset],
        pca_datasets = [dataset],
        ).data:
        fig.add_trace(
            data, row=(i // 4) + 1, col=(i % 4) + 1
        )

for row in range(3):
    for col in range(4):
        fig.update_yaxes(
            scaleanchor = f"x{row * 4 + col + 1}",
            scaleratio = 1,
            row=row+1,
            col=col+1,
        )

fig.update_coloraxes(
    colorscale = 'Bluered_r'
)

fig.update_layout(
    height=1000,
    width=1000,
    showlegend=False,
    title = {
        'text' : 'PCA visualizations of all datasets',
        'font' : {'size' : 30}
    },
    coloraxis_showscale=False,
)

In [ ]:
# does the human/animal separation found in one dataset's PCA basis carry over to the others?
pca_datasets = [
    'human_farmed_neutral',
    'human_wild_neutral',
    'human_farmed_harm',
]

plot_datasets = [
    'human_farmed_neutral',
    'human_wild_neutral',
    'human_farmed_harm',
    'human_wild_harm',
    'human_companion_harm',
]

fig = make_subplots(rows=len(pca_datasets), cols=len(plot_datasets), 
    vertical_spacing=0.05, horizontal_spacing=0.01,
    )

for col, plot_dataset in enumerate(plot_datasets):
    fig.update_xaxes(title= {
                    'text': plot_dataset,
                    'font' : {'size' : 20}
                    },
                    row=len(pca_datasets), col=col+1
                )
for row, pca_dataset in enumerate(pca_datasets):
    fig.update_yaxes(title= {
                    'text': pca_dataset,
                    'font' : {'size' : 20}
                    },
                    row=row+1, col=1
                )

td = TruthData.from_datasets(plot_datasets, model=model, layer=layer, center=True, noperiod=noperiod, device=device)

for row, pca_dataset in enumerate(pca_datasets):
    for col, plot_dataset in enumerate(plot_datasets):
        subfig = td.plot(
            dimensions = 2,
            plot_datasets = [plot_dataset],
            pca_datasets = [pca_dataset],
            color='label',
        )
        fig.add_trace(subfig.data[0], row=row+1, col=col+1)

for row in range(len(pca_datasets)):
    for col in range(len(plot_datasets)):
        fig.update_yaxes(
            scaleanchor = f"x{row * len(plot_datasets) + col + 1}",
            scaleratio = 1,
            row=row+1,
            col=col+1,
        )

fig.update_coloraxes(
    colorscale = 'Bluered_r'
)

fig.update_layout(
    height=900,
    width=1500,
    coloraxis_showscale=False,
    title = {
        'text': f'Dataset visualizations in various PCA bases',
        'font' : {'size' : 30}
    }
)

fig.show()

In [ ]:
from plotly.subplots import make_subplots

# harm + neutral versions of each contrast, colored by (dataset, label):
# does the category separation line up across the two contexts?
pairs = [
    ['human_farmed_harm', 'human_farmed_neutral'],
    ['human_wild_harm', 'human_wild_neutral'],
    ['human_companion_harm', 'human_companion_neutral'],
]

# colormappings under the Rainbow colorscale
RED = 1
BLUE = .17
PURPLE = 0
YELLOW = .73

fig = make_subplots(rows=1, cols=len(pairs),
                    shared_yaxes=True,
                    x_title='PC1', y_title='PC2',
                    subplot_titles=[ '+'.join([dataset for dataset in pair]) for pair in pairs]
                    )

for i, pair in enumerate(pairs):
    td = TruthData.from_datasets(
        pair,
        model=model,
        layer=layer,
        noperiod=noperiod,
        device=device
    )

    
    td.df.loc[pair[0], 'label'] = td.df.loc[pair[0]]['label'].apply(lambda x: BLUE if x == 1 else RED)
    td.df.loc[pair[1], 'label'] = td.df.loc[pair[1]]['label'].apply(lambda x: YELLOW if x == 1 else PURPLE)


    subfig = td.plot(
        dimensions=2,
        color = 'label',
    )

    fig.add_trace(subfig.data[0], row=1, col=i+1)

fig.update_coloraxes(
    colorscale='Rainbow'
)

fig.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# how the category separation develops across layers (requires acts cached at
# these layers, e.g. generate_acts.py --layers -1)
datasets = ['human_farmed_neutral', 'human_farmed_harm', 'farmed_wild_harm']
layers = [4, 8, 12, 16, 24]

figs = [[] for _ in datasets]

for i, dataset in enumerate(datasets):
    for layer_ in layers:
        fig = TruthData.from_datasets(
            [dataset],
            model=model,
            layer=layer_,
            noperiod=noperiod,
            device=device
            ).plot(
                dimensions=2,
                color='label',
            )
        figs[i].append(fig)

fig = make_subplots(rows = len(datasets), cols = len(layers),
                    subplot_titles=[f"layer {layer_}" for layer_ in layers],
                    vertical_spacing=0.05)

for i, dataset in enumerate(datasets):
    for j, layer_ in enumerate(layers):
        for data in figs[i][j].data:
            data['showlegend'] = False
            fig.add_trace(data, row=i+1, col=j+1)

for i, dataset in enumerate(datasets):
    fig.update_yaxes(title_text=dataset, row=i+1, col=1)

fig.update_coloraxes(
    colorscale='Bluered_r'
)

fig.update_layout(height=800, width=1200, coloraxis_showscale=False)

fig.update_layout(
    title = {
        'text' : f"Dataset visualizations across layers",
        'font' : {'size' : 20},
    }
)

# fig.write_image('junk/layer_sweep.png')  # needs junk/ to exist and kaleido installed

fig.show()